# 08 — Perturbation Effects

**Prerequisites:** Run [07_normalization_dimred.ipynb](07_normalization_dimred.ipynb) first. Requires `data/replogle2022_k562_essential.h5ad` (downloaded in notebook 03).

**What you'll learn:**
- Why pseudobulk DE is the statistically correct approach (vs. per-cell Wilcoxon)
- How to construct pseudobulk profiles from single-cell data
- Running PyDESeq2 for each perturbation vs. NTC
- Summarizing perturbation effects: DEG counts, LFC distributions
- `pertpy` E-distance as a non-parametric complement
- Ranking perturbations by transcriptional effect size

**Dataset:** Replogle et al. 2022 K562 Essential — ~2,057 CRISPRi perturbations, ~5 guides/gene. This is a genome-scale screen, making it ideal for ranking and comparing perturbation effects at scale.

**Estimated time:** 3 hours (PyDESeq2 across 2,000+ perturbations is computationally intensive; a small subset is processed first)

In [ ]:
import os
import pickle
import scanpy as sc
import pertpy as pt
import anndata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor="white")

DATA_DIR = "data"
FIG_DIR = "figures"

# Load Replogle 2022 in backed mode to conserve RAM
REPLOGLE_PATH = os.path.join(DATA_DIR, "replogle2022_k562_essential.h5ad")
adata = sc.read_h5ad(REPLOGLE_PATH, backed="r")
print(adata)
print("\nobs columns:", adata.obs.columns.tolist())

## 1. Why pseudobulk? The pseudoreplication problem

If you run a Wilcoxon test comparing 300 perturbed cells to 5,000 NTC cells, the p-values will be extremely small — but this is statistically invalid. Single cells from the same transduced well are **not independent observations**: they share all sources of technical noise (sequencing batch, ambient RNA, etc.).

**Pseudoreplication** = treating non-independent observations as independent. In single-cell DE, this inflates sample size by 100–500× and produces catastrophically low p-values.

**Pseudobulk** is the fix: aggregate (sum) raw UMI counts from all cells in the same perturbation group and the same biological replicate into one bulk-like "pseudo-sample". Then run a bulk RNA-seq DE method (DESeq2, edgeR) on these pseudo-samples. You need ≥2 pseudo-replicates per condition (guide) for variance estimation.

The Replogle dataset has multiple guides per gene, which serve as biological replicates.

In [ ]:
# Identify perturbation column in Replogle dataset
for col in adata.obs.columns:
    n = adata.obs[col].nunique()
    if 1000 < n < 20000:
        print(f"Candidate perturbation column: '{col}' ({n} unique)")

In [ ]:
# Set these based on the output above
# Common Replogle column names:
GENE_COL = None
GUIDE_COL = None

for candidate_gene in ["gene", "gene_target", "target_gene", "perturbation"]:
    if candidate_gene in adata.obs.columns:
        GENE_COL = candidate_gene
        break

for candidate_guide in ["guide_ids", "guide", "sgRNA", "guide_id"]:
    if candidate_guide in adata.obs.columns:
        GUIDE_COL = candidate_guide
        break

if GENE_COL is None:
    # Fallback: use the first categorical column
    GENE_COL = adata.obs.select_dtypes(include=["category", "object"]).columns[0]

print(f"Gene column: {GENE_COL}")
print(f"Guide column: {GUIDE_COL}")

# Identify NTC identifier in this dataset
ntc_values = [x for x in adata.obs[GENE_COL].unique()
              if any(kw in str(x).lower() for kw in ["ctrl", "control", "non", "ntc", "safe"])]
print(f"NTC identifiers: {ntc_values[:5]}")

## 2. Working subset

Running PyDESeq2 on all 2,057 perturbations takes several hours. We first demonstrate the pipeline on a subset of 20 genes + NTC, then provide a loop for the full dataset.

In [ ]:
# Load a subset into memory (backed mode doesn't support slicing well)
# Select NTC + 20 target genes with the most cells
adata_mem = adata.to_memory()  # Load full dataset to memory
print(f"Loaded: {adata_mem.shape}")

# Select top 20 perturbations by cell count
gene_counts = adata_mem.obs[GENE_COL].value_counts()
ntc_labels = [x for x in gene_counts.index
              if any(kw in str(x).lower() for kw in ["ctrl", "control", "non", "ntc"])]

top_genes = [g for g in gene_counts.index[:25] if g not in ntc_labels][:20]
selected_genes = ntc_labels + top_genes

subset_mask = adata_mem.obs[GENE_COL].isin(selected_genes)
adata_sub = adata_mem[subset_mask].copy()
print(f"Working subset: {adata_sub.shape}, {adata_sub.obs[GENE_COL].nunique()} perturbations")

## 3. Pseudobulk aggregation

In [ ]:
def pseudobulk(adata, group_col, raw_layer="counts"):
    """
    Aggregate single-cell counts into pseudobulk profiles.
    
    Each group (e.g., one guide × one replicate) becomes one row
    in the output, with gene counts summed across all cells in the group.
    
    Returns: pd.DataFrame (groups × genes) of integer counts
    """
    # Get raw counts
    if raw_layer in adata.layers:
        X = adata.layers[raw_layer]
    elif adata.raw is not None:
        X = adata.raw.to_adata().X
    else:
        X = adata.X
    
    if scipy.sparse.issparse(X):
        X = X.toarray()
    
    obs = adata.obs[[group_col]].copy()
    obs["_row"] = np.arange(len(obs))
    
    results = []
    for group, idx in obs.groupby(group_col)["_row"]:
        summed = X[idx.values].sum(axis=0)  # sum across cells
        results.append({"group": group, **dict(zip(adata.var_names, summed))})
    
    pb = pd.DataFrame(results).set_index("group")
    return pb.astype(int)


# Pseudobulk by guide (each guide = one replicate for its target gene)
group_col = GUIDE_COL if GUIDE_COL else GENE_COL
print(f"Aggregating by '{group_col}'...")

pb = pseudobulk(adata_sub, group_col=group_col)
print(f"Pseudobulk matrix: {pb.shape} (groups × genes)")
print(pb.iloc[:5, :5])

In [ ]:
# Add metadata: map each guide back to its target gene
if GUIDE_COL:
    guide_to_gene = adata_sub.obs.set_index(GUIDE_COL)[GENE_COL].to_dict()
    pb_meta = pd.DataFrame({
        "guide_id": pb.index,
        "gene_target": [guide_to_gene.get(g, g) for g in pb.index],
    })
    pb_meta["is_ntc"] = pb_meta["gene_target"].isin(ntc_labels)
else:
    pb_meta = pd.DataFrame({
        "guide_id": pb.index,
        "gene_target": pb.index,
        "is_ntc": pb.index.isin(ntc_labels),
    })

print("Pseudobulk metadata:")
print(pb_meta.head(10))
print(f"\nNTC pseudo-samples: {pb_meta['is_ntc'].sum()}")
print(f"Treatment pseudo-samples: {(~pb_meta['is_ntc']).sum()}")

## 4. PyDESeq2 — one perturbation vs. NTC

In [ ]:
def run_deseq2_one_perturbation(pb_counts, pb_meta, target_gene, ntc_labels, min_count=5):
    """
    Run PyDESeq2 for one target gene vs. NTC.
    
    Returns a pd.DataFrame with columns: gene, log2FoldChange, pvalue, padj
    """
    # Select rows: target gene guides + NTC guides
    target_rows = pb_meta[pb_meta["gene_target"] == target_gene].index
    ntc_rows = pb_meta[pb_meta["is_ntc"]].index
    
    if len(target_rows) < 2 or len(ntc_rows) < 2:
        return None  # Need at least 2 samples per group
    
    selected = list(target_rows) + list(ntc_rows)
    counts_sub = pb_counts.loc[selected].copy()
    
    # Filter low-count genes
    keep_genes = counts_sub.sum(axis=0) >= min_count
    counts_sub = counts_sub.loc[:, keep_genes]
    
    meta_sub = pd.DataFrame({
        "condition": ["treatment"] * len(target_rows) + ["control"] * len(ntc_rows)
    }, index=selected)
    
    try:
        dds = DeseqDataSet(
            counts=counts_sub,
            metadata=meta_sub,
            design="~condition",
            quiet=True,
        )
        dds.deseq2()
        
        stat_res = DeseqStats(dds, contrast=("condition", "treatment", "control"), quiet=True)
        stat_res.summary()
        
        results = stat_res.results_df.reset_index()
        results.columns = [c if c != "index" else "gene" for c in results.columns]
        results["gene_target"] = target_gene
        return results
        
    except Exception as exc:
        print(f"  DESeq2 failed for {target_gene}: {exc}")
        return None


# Test on one perturbation
test_gene = top_genes[0]
print(f"Running DESeq2 for {test_gene} vs. NTC...")
result_one = run_deseq2_one_perturbation(pb, pb_meta, test_gene, ntc_labels)
if result_one is not None:
    print(f"Result shape: {result_one.shape}")
    sig = result_one[result_one["padj"] < 0.05]
    print(f"Significant DEGs (padj < 0.05): {len(sig)}")
    print(result_one.sort_values("padj").head(5)[["gene", "log2FoldChange", "pvalue", "padj"]])

In [ ]:
# Run for all genes in the subset
all_results = []
target_genes_to_test = [g for g in pb_meta["gene_target"].unique() if g not in ntc_labels]

print(f"Running DESeq2 for {len(target_genes_to_test)} perturbations...")
for i, gene in enumerate(target_genes_to_test):
    if i % 5 == 0:
        print(f"  {i+1}/{len(target_genes_to_test)}: {gene}")
    result = run_deseq2_one_perturbation(pb, pb_meta, gene, ntc_labels)
    if result is not None:
        all_results.append(result)

de_results = pd.concat(all_results, ignore_index=True)
print(f"\nTotal DE results rows: {len(de_results)}")
print(de_results.head())

In [ ]:
# Summarize: DEG counts per perturbation
deg_summary = de_results.groupby("gene_target").apply(
    lambda df: pd.Series({
        "n_sig_up": ((df["padj"] < 0.05) & (df["log2FoldChange"] > 0.5)).sum(),
        "n_sig_down": ((df["padj"] < 0.05) & (df["log2FoldChange"] < -0.5)).sum(),
        "mean_abs_lfc": df.loc[df["padj"] < 0.05, "log2FoldChange"].abs().mean(),
    })
).fillna(0)
deg_summary["n_sig_total"] = deg_summary["n_sig_up"] + deg_summary["n_sig_down"]
deg_summary = deg_summary.sort_values("n_sig_total", ascending=False)

print("Top perturbations by number of significant DEGs:")
print(deg_summary.head(10))

In [ ]:
# Visualize: perturbations ranked by effect size
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: n_sig_total per perturbation
axes[0].barh(
    range(len(deg_summary)),
    deg_summary["n_sig_total"],
    color="steelblue", edgecolor="none"
)
axes[0].set_yticks(range(len(deg_summary)))
axes[0].set_yticklabels(deg_summary.index, fontsize=8)
axes[0].set_xlabel("Significant DEGs (|LFC| > 0.5, padj < 0.05)")
axes[0].set_title("Perturbations ranked by DEG count")

# Scatter: n_sig_up vs. n_sig_down
axes[1].scatter(deg_summary["n_sig_up"], deg_summary["n_sig_down"],
                c="steelblue", alpha=0.7, s=50)
for gene, row in deg_summary.head(5).iterrows():
    axes[1].annotate(gene, (row["n_sig_up"], row["n_sig_down"]),
                     fontsize=8, ha="left")
axes[1].set_xlabel("Significant upregulated DEGs")
axes[1].set_ylabel("Significant downregulated DEGs")
axes[1].set_title("Up vs. down DEGs per perturbation")
axes[1].axline((0, 0), slope=1, color="gray", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "08_perturbation_effects.png"), dpi=150, bbox_inches="tight")
plt.show()

## 5. Pertpy E-distance

The energy distance (E-distance) measures how much the transcriptome **distribution** of perturbed cells differs from NTC cells. Unlike pseudobulk DE, it is non-parametric and does not require pre-selection of individual genes.

It is particularly useful for:
- Ranking perturbations by overall transcriptional impact
- Detecting subtle global shifts that do not rise to significance in gene-by-gene tests

In [ ]:
# Load Norman processed data for E-distance demo
norman_path = os.path.join(DATA_DIR, "norman_processed.h5ad")
if os.path.exists(norman_path):
    norman = sc.read_h5ad(norman_path)
    PERT_COL = "perturbation" if "perturbation" in norman.obs.columns else "condition"
    
    # Use PCA representation
    if "X_pca" in norman.obsm:
        print("Computing E-distance on Norman dataset (PCA representation)...")
        try:
            dist = pt.tl.Distance(metric="edistance", obsm_key="X_pca")
            
            # Compute pairwise distances between perturbations
            ntc_label = [x for x in norman.obs[PERT_COL].unique()
                         if any(kw in str(x).lower() for kw in ["ctrl","control","ntc"])][0]
            
            edist_results = dist.onesided_distances(
                norman, groupby=PERT_COL, selected_group=ntc_label, show_progress=True
            )
            print("E-distance results:")
            print(edist_results.sort_values(ascending=False).head(10))
            
        except Exception as e:
            print(f"pertpy Distance computation raised: {e}")
            print("This may be a version compatibility issue — check pertpy documentation.")
else:
    print("Norman processed file not found. Run notebook 07 first.")

## 6. Save results

In [ ]:
# Save DE results
results_path = os.path.join(DATA_DIR, "de_results.pkl")
with open(results_path, "wb") as f:
    pickle.dump(de_results, f)
print(f"DE results saved to {results_path}")

# Save perturbation summary
summary_path = os.path.join(DATA_DIR, "perturbation_scores.csv")
deg_summary.to_csv(summary_path)
print(f"Perturbation scores saved to {summary_path}")

## Key takeaways

1. **Always use pseudobulk DE** for Perturb-seq — per-cell Wilcoxon inflates type I error by treating cells as independent
2. Pseudobulk construction: sum raw counts per (guide × replicate), then apply DESeq2 or edgeR
3. The Replogle dataset's 5 guides/gene provide natural biological replicates for pseudobulk
4. E-distance is a useful complement: model-free, captures global distribution shifts, good for ranking
5. Perturbations with many DEGs are not necessarily the most biologically interesting — specificity (few, interpretable DEGs in a clear pathway) is often more valuable than total DEG count

**Next:** [09_mixscape.ipynb](09_mixscape.ipynb) and/or [10_visualization.ipynb](10_visualization.ipynb)